# 🛒 Segmentasi Pelanggan dengan K-Means Clustering
### Dataset: Wholesale Customers (UCI Machine Learning Repository)

---

**Nama:** Zahrotun Nafisah  
**NIM:** 21422008  
**Prodi:** Sistem Informasi 2022  
**Mata Kuliah:** Business Intelligence  

---

## 📌 Deskripsi Proyek

Proyek ini bertujuan mengelompokkan pelanggan grosir berdasarkan pola pengeluaran tahunan mereka menggunakan algoritma **K-Means Clustering**. Hasil segmentasi diharapkan dapat mendukung pengambilan keputusan bisnis berbasis data (*Business Intelligence*).

**Dataset:** [Wholesale Customers Data Set - UCI ML Repository](https://archive.ics.uci.edu/ml/datasets/Wholesale+customers)  
**Ukuran:** 440 baris × 8 kolom  
**Atribut:** Channel, Region, Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicassen

---
## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ Semua library berhasil diimport!')

---
## 2. Load & Eksplorasi Dataset

In [ ]:
# Load dataset
df = pd.read_csv('Wholesale_customers_data.csv')

print('📊 Shape dataset:', df.shape)
print('\n📋 5 baris pertama:')
df.head()

In [ ]:
# Informasi dataset
print('📌 Info Dataset:')
df.info()

In [ ]:
# Statistik deskriptif
print('📈 Statistik Deskriptif:')
df.describe().round(2)

In [ ]:
# Cek missing values
print('🔍 Missing Values per Kolom:')
missing = df.isnull().sum()
print(missing)
print(f'\n✅ Total missing values: {missing.sum()} (tidak ada missing value!)')

In [ ]:
# Distribusi setiap fitur
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribusi Pengeluaran per Kategori Produk', fontsize=16, fontweight='bold')

for i, (ax, feat) in enumerate(zip(axes.flatten(), features)):
    ax.hist(df[feat], bins=30, edgecolor='white', color=sns.color_palette('Set2')[i])
    ax.set_title(feat, fontsize=13, fontweight='bold')
    ax.set_xlabel('Pengeluaran Tahunan')
    ax.set_ylabel('Frekuensi')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('distribusi_fitur.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Catatan: Data memiliki distribusi right-skewed — normalisasi sangat diperlukan.')

In [ ]:
# Heatmap korelasi
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Korelasi Antar Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('korelasi_fitur.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Grocery & Detergents_Paper memiliki korelasi tinggi — keduanya sering dibeli bersamaan.')

---
## 3. Preprocessing Data

In [ ]:
# Seleksi fitur numerik (pola belanja)
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X = df[features].copy()

# Normalisasi menggunakan StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

print('✅ Normalisasi selesai!')
print('\n📋 Contoh data setelah normalisasi (5 baris pertama):')
print(X_scaled_df.head().round(4))
print(f'\n📊 Mean setelah scaling: {X_scaled_df.mean().round(6).values} (mendekati 0 ✅)')
print(f'📊 Std setelah scaling:  {X_scaled_df.std().round(6).values} (mendekati 1 ✅)')

---
## 4. Menentukan Jumlah Klaster Optimal — Elbow Method

In [ ]:
# Elbow Method
wcss = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300,
                    n_init=10, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Visualisasi Elbow
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, wcss, marker='o', color='steelblue', linewidth=2.5,
        markersize=8, markerfacecolor='white', markeredgewidth=2)
ax.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='K optimal = 3')
ax.scatter([3], [wcss[2]], color='red', s=120, zorder=5)
ax.set_title('Elbow Method untuk Menentukan Jumlah Klaster Optimal', fontsize=14, fontweight='bold')
ax.set_xlabel('Jumlah Cluster (K)', fontsize=12)
ax.set_ylabel('WCSS (Within-Cluster Sum of Squares)', fontsize=12)
ax.legend(fontsize=11)
ax.set_xticks(K_range)

plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 Titik "siku" (elbow) berada di K=3 → dipilih sebagai jumlah klaster optimal.')

---
## 5. Training Model K-Means (K=3)

In [ ]:
# Training K-Means dengan K=3
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300,
                n_init=10, random_state=42)
kmeans.fit(X_scaled)

# Tambahkan label klaster ke dataframe
df['Cluster'] = kmeans.labels_

print('✅ Model K-Means berhasil dilatih!')
print('\n📊 Distribusi pelanggan per klaster:')
cluster_counts = df['Cluster'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    pct = count / len(df) * 100
    print(f'   Klaster {cluster}: {count} pelanggan ({pct:.1f}%)')

---
## 6. Evaluasi Model — Silhouette Score

In [ ]:
# Hitung Silhouette Score
sil_score = silhouette_score(X_scaled, kmeans.labels_)

print(f'📏 Silhouette Score untuk K=3: {sil_score:.3f}')
print()

# Interpretasi
if sil_score >= 0.7:
    interp = 'Sangat Baik — klaster sangat terpisah jelas.'
elif sil_score >= 0.5:
    interp = 'Cukup Baik — struktur klaster terbentuk jelas dan tidak terlalu tumpang tindih.'
elif sil_score >= 0.25:
    interp = 'Lemah — struktur klaster kurang jelas.'
else:
    interp = 'Buruk — mungkin data tidak cocok untuk clustering.'

print(f'💡 Interpretasi: {interp}')

# Bandingkan silhouette score untuk berbagai K
print('\n📊 Perbandingan Silhouette Score untuk K=2 sampai K=6:')
for k in range(2, 7):
    km_temp = KMeans(n_clusters=k, init='k-means++', n_init=10,
                     random_state=42).fit(X_scaled)
    s = silhouette_score(X_scaled, km_temp.labels_)
    marker = ' ← dipilih' if k == 3 else ''
    print(f'   K={k}: {s:.3f}{marker}')

---
## 7. Visualisasi Hasil Klaster (PCA 2D)

In [ ]:
# Reduksi dimensi dengan PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'📊 Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}')
print(f'📊 Total variance explained: {sum(pca.explained_variance_ratio_):.2%}')

# Visualisasi scatter plot PCA
colors = ['#2196F3', '#FF5722', '#4CAF50']
labels = ['Klaster 0', 'Klaster 1', 'Klaster 2']

fig, ax = plt.subplots(figsize=(10, 7))

for cluster in range(3):
    mask = df['Cluster'] == cluster
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=colors[cluster], label=labels[cluster],
               alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

# Plot centroid
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           marker='*', s=300, c='black', label='Centroid', zorder=5)

ax.set_title('Visualisasi Hasil Klaster Pelanggan (PCA 2D)', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('pca_cluster_viz.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Analisis Karakteristik Tiap Klaster

In [ ]:
# Ringkasan rata-rata pengeluaran per klaster
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
summary = df.groupby('Cluster')[features].mean().round(0).astype(int)
summary['Jumlah Pelanggan'] = df.groupby('Cluster').size()

print('📋 Ringkasan Karakteristik Tiap Klaster:')
print(summary.to_string())

In [ ]:
# Bar chart rata-rata pengeluaran per klaster
fig, ax = plt.subplots(figsize=(13, 6))

x = np.arange(len(features))
width = 0.25
cluster_colors = ['#2196F3', '#FF5722', '#4CAF50']

for i, (cluster, color) in enumerate(zip(range(3), cluster_colors)):
    vals = summary.loc[cluster, features].values
    bars = ax.bar(x + i * width, vals, width,
                  label=f'Klaster {cluster} (n={summary.loc[cluster, "Jumlah Pelanggan"]})',
                  color=color, alpha=0.85, edgecolor='white')

ax.set_title('Rata-rata Pengeluaran per Kategori Produk per Klaster', fontsize=14, fontweight='bold')
ax.set_xlabel('Kategori Produk', fontsize=12)
ax.set_ylabel('Rata-rata Pengeluaran (Satuan Moneter)', fontsize=12)
ax.set_xticks(x + width)
ax.set_xticklabels(features, fontsize=11)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('cluster_spending.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Radar chart profil klaster (normalized)
from matplotlib.patches import FancyBboxPatch
from math import pi

categories = features
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw=dict(polar=True))
fig.suptitle('Profil Pengeluaran Tiap Klaster (Radar Chart)', fontsize=14, fontweight='bold')

cluster_names = ['Klaster 0\n(High Grocery)', 'Klaster 1\n(Mayoritas - Medium)', 'Klaster 2\n(VIP - Ultra High)']

# Normalize per feature (0-1)
norm_summary = summary[features].copy()
for col in features:
    norm_summary[col] = (norm_summary[col] - norm_summary[col].min()) / \
                        (norm_summary[col].max() - norm_summary[col].min() + 1e-9)

for idx, (ax, cluster, color, name) in enumerate(zip(axes, range(3), cluster_colors, cluster_names)):
    values = norm_summary.loc[cluster].tolist()
    values += values[:1]

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=9)
    ax.set_ylim(0, 1)
    ax.plot(angles, values, color=color, linewidth=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_title(name, size=11, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Business Insight & Rekomendasi Strategi

In [ ]:
print('=' * 65)
print('       BUSINESS INSIGHT & REKOMENDASI STRATEGI')
print('=' * 65)

insights = [
    {
        'klaster': 'Klaster 0 — Pelanggan Grocery & Retail',
        'jumlah': summary.loc[0, 'Jumlah Pelanggan'],
        'karakteristik': [
            'Pengeluaran tinggi pada Grocery, Milk, dan Detergents_Paper',
            'Tipikal pelanggan retail atau minimarket',
        ],
        'strategi': [
            'Buat program loyalitas B2B atau diskon grosir khusus',
            'Tawarkan kontrak pembelian jangka panjang',
            'Fokus pada efisiensi suplai produk grocery & household',
        ]
    },
    {
        'klaster': 'Klaster 1 — Pelanggan Umum (Mayoritas)',
        'jumlah': summary.loc[1, 'Jumlah Pelanggan'],
        'karakteristik': [
            'Pengeluaran sedang dan merata di semua kategori',
            'Merupakan segmen terbesar (mayoritas pelanggan)',
        ],
        'strategi': [
            'Kampanye promosi reguler dan diskon bundling produk',
            'Edukasi produk untuk mendorong upselling',
            'Program loyalitas dengan insentif tier (poin, cashback)',
        ]
    },
    {
        'klaster': 'Klaster 2 — Pelanggan VIP (Ultra High Spender)',
        'jumlah': summary.loc[2, 'Jumlah Pelanggan'],
        'karakteristik': [
            'Pengeluaran ekstrem tinggi di Fresh, Frozen, dan Delicassen',
            'Kemungkinan besar restoran fine dining atau hotel bintang',
        ],
        'strategi': [
            'Berikan layanan personal/VIP dan account manager khusus',
            'Tawarkan produk eksklusif, impor, atau private label',
            'Pertahankan relasi dengan pendekatan langsung dan kontrak premium',
        ]
    }
]

for ins in insights:
    print(f"\n🎯 {ins['klaster']} ({ins['jumlah']} pelanggan)")
    print('   Karakteristik:')
    for k in ins['karakteristik']:
        print(f'   • {k}')
    print('   Strategi yang Disarankan:')
    for s in ins['strategi']:
        print(f'   → {s}')

print('\n' + '=' * 65)

---
## 10. Export Hasil Segmentasi

In [ ]:
# Export data dengan label klaster ke CSV
df.to_csv('wholesale_customers_segmented.csv', index=False)

print('✅ File hasil segmentasi berhasil disimpan: wholesale_customers_segmented.csv')
print(f'\n📊 Preview hasil:')
df[['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen', 'Cluster']].head(10)

---
## 11. Kesimpulan

Algoritma **K-Means Clustering** berhasil mengelompokkan **440 pelanggan grosir** menjadi **3 segmen** berdasarkan pola pengeluaran tahunan mereka:

| Klaster | Profil | Jumlah | Strategi Utama |
|---------|--------|--------|----------------|
| **0** | Pelanggan Grocery & Retail | 45 | Program loyalitas B2B & kontrak jangka panjang |
| **1** | Pelanggan Umum (Mayoritas) | 393 | Promosi reguler & bundling produk |
| **2** | Pelanggan VIP (Ultra High) | 2 | Layanan personal & produk eksklusif |

**Evaluasi Model:**
- ✅ Silhouette Score: **0.548** (cukup baik — struktur klaster jelas)
- ✅ Elbow Method menunjukkan K=3 sebagai titik optimal
- ✅ Visualisasi PCA 2D mengkonfirmasi pemisahan klaster

**Saran Pengembangan:**
1. Tambahkan fitur RFM (Recency, Frequency, Monetary) untuk segmentasi yang lebih kaya
2. Coba bandingkan dengan metode lain: Hierarchical Clustering atau DBSCAN
3. Integrasikan hasil ke dashboard BI interaktif (Power BI / Tableau)
4. Pertimbangkan penanganan outlier sebelum clustering untuk hasil yang lebih robust

---
## Referensi
- Dua, D. & Graff, C. (2021). *Wholesale Customers Data Set*. UCI Machine Learning Repository. https://archive.ics.uci.edu/ml/datasets/Wholesale+customers
- Scikit-learn Documentation: https://scikit-learn.org/stable/modules/clustering.html